# PickPOV · Week 1: get the data

This notebook:
1. Connects to Google Drive, where the project and VODs live
2. Gets one Summit VOD (download, or upload one you downloaded on your PC)
3. Saves sample frames
4. Calibrates the screen regions (minimap, HUD, kill feed, POV name)

**Before you start:** upload the whole `PickPOV` folder to `My Drive/PickPOV`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/PickPOV'
%cd {PROJECT}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, f'{PROJECT}/src')

## 2. Pick a Summit map

All 18 Summit maps from Champions 2026 are in `data/champions2026_summit_maps.csv`. Fill in `vod_url` as you find each VOD on the official VALORANT Champions Tour YouTube channel.

In [ ]:
import pandas as pd
maps = pd.read_csv('data/champions2026_summit_maps.csv')
maps[['date', 'stage', 'team_a', 'team_b', 'map_number', 'score_a', 'score_b']]

## 3. Get one VOD

**Option A — download here.** YouTube sometimes blocks Colab with a "confirm you're not a bot" error. If that happens, use option B.

Set `START`/`END` to just the Summit map so you don't download the whole series.

In [ ]:
URL = ''            # paste the YouTube link
START, END = '', '' # e.g. '0:12:00', '0:55:00'; leave empty for the whole video
NAME = '2026-10-18_loud_vs_team-vitality_map1'

if URL:
    args = f'--name {NAME}' + (f' --start {START}' if START else '') + (f' --end {END}' if END else '')
    !python scripts/download_vod.py "{URL}" {args}

**Option B — download on your PC.** In the `PickPOV` folder on your computer:

```
pip install -r requirements.txt
python scripts/download_vod.py <URL> --start 0:12:00 --end 0:55:00 --name 2026-10-18_loud_vs_team-vitality_map1
```

Then upload `vods/2026-10-18_loud_vs_team-vitality_map1.mp4` to `My Drive/PickPOV/vods/`.

In [ ]:
from pickpov.video import video_info
VIDEO = f'vods/{NAME}.mp4'
video_info(VIDEO)

## 4. Save sample frames

In [ ]:
from pickpov.video import save_frames
saved = save_frames(VIDEO, f'frames/{NAME}', every_s=20, max_frames=40)
print(len(saved), 'frames saved')

In [ ]:
import cv2, matplotlib.pyplot as plt

def show(img, title=''):
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis('off'); plt.show()

frame = cv2.imread(str(saved[len(saved) // 2]))  # a frame from the middle
show(frame, saved[len(saved) // 2].name)

## 5. Calibrate regions

Pick a frame that shows **live gameplay** (minimap, HUD and kill feed visible). Read the box positions off the grid, edit `configs/broadcast_regions.yaml`, and re-run the last cell until every yellow box fits.

In [ ]:
from pickpov.regions import draw_grid
show(draw_grid(frame, step=0.05), 'grid: numbers are fractions of width / height')

In [ ]:
from pickpov.regions import load_regions, draw_regions
regions = load_regions('configs/broadcast_regions.yaml')
show(draw_regions(frame, regions), 'regions')
for name, r in regions.items():
    show(r.crop(frame), name)

## Done when

- [ ] One Summit VOD is in `vods/`
- [ ] Every region box fits on 3 different gameplay frames
- [ ] You've noted what replays and caster cams look like (next step: detect and skip them)